In [4]:
# Save file list for PM2.5

In [5]:
import os
import glob
import json
from datetime import datetime
from utils.utils import get_scenario_config

In [6]:
# Create a config variable to store file path information for each model/scenario
# Adjust as necessary for new data
MODEL_PATH_CONFIG = {
    # CESM2
    "CESM2": {
        "base": "/glade/campaign",
        "scenarios": {
            # ARISE-1.5K-SAI
            "ARISE": {
                "dir_template": (
                    "{base}/cesm/collections/ARISE-SAI-1.5/"
                    "b.e21.BW.f09_g17.SSP245-TSMLT-GAUSS-DEFAULT.0{ens}/"
                    "atm/proc/tseries/month_1/"
                ),
                "search_term": "*.PM25.*.nc",
            },

            # SSP2-4.5
            "SSP245": {
                "dir_template": (
                    "{base}/cesm/collections/CESM2-WACCM-SSP245/"
                    "b.e21.BWSSP245cmip6.f09_g17.CMIP6-SSP2-4.5-WACCM.0{ens}/"
                    "atm/proc/tseries/month_1/"
                ),
                "search_term": "*.PM25.*.nc",
            },

            # SSP2-4.5 (same as above but is only for 3 ensemble members)
            "SSP245_G6": {
                "dir_template": (
                    "{base}/cesm/collections/CESM2-WACCM-SSP245/"
                    "b.e21.BWSSP245cmip6.f09_g17.CMIP6-SSP2-4.5-WACCM.0{ens}/"
                    "atm/proc/tseries/month_1/"
                ),
                "search_term": "*.PM25.*.nc",
            },

            # Historical
            "hist": {
                "dir_template": (
                    "{base}/cesm/development/wawg/WACCM6-TSMLT-HIST/"
                    "b.e21.BWHISTcmip6.f09_g17.CMIP6-historical-WACCM.1980_2014.0{ens}/"
                    "atm/proc/tseries/month_1/"
                ),
                "search_term": "*.PM25.*.nc",
            },

            # G6-1.5K-SAI
            "G6-1.5K": {
                "dir_template": (
                    "{base}/collections/rda/data/d651059/ARISE-SAI-1.5/"
                    "b.e21.BW.f09_g17.SSP245-G6-1p5K-SAI.0{ens}/"
                    "atm/proc/tseries/month_1/"
                ),
                "search_term": "*.PM25.*.nc",
            },
        },
    },
}


# return dir and search term for model/scenario
def get_path_config(model, scenario):
    try:
        entry = MODEL_PATH_CONFIG[model]
        scen = entry["scenarios"][scenario]
        return {
            "dir_template": scen["dir_template"].format(base=entry["base"], ens="{ens}"),
            "search_term": scen["search_term"],
        }
    except KeyError:
        available_models = list(MODEL_PATH_CONFIG.keys())
        available_scenarios = (
            list(MODEL_PATH_CONFIG[model]["scenarios"].keys()) if model in MODEL_PATH_CONFIG else []
        )
        raise ValueError(
            f"Config not found for model '{model}', scenario '{scenario}'.\n"
            f"Available models: {available_models}\n"
            f"Available scenarios for {model if model in MODEL_PATH_CONFIG else 'N/A'}: {available_scenarios}"
        )

In [7]:
def save_file_list_with_metadata(model, scenario, ens_num, file_list, DIR, filename):
    data = {
        "model": model,
        "scenario": scenario,
        "ensemble_number": ens_num,
        "generated_on": datetime.now().isoformat(),
        "files": file_list
    }
    file_path = os.path.join(DIR, filename)
    with open(file_path, "w") as f:
        json.dump(data, f, indent=2)

In [10]:
# === Scenario and path config ===
# Set to whatever scenario and model you want
# Function returns error if not recognised
model = "CESM2"
scenario = "G6-1.5K"

config = get_scenario_config(model, scenario)
ensemble_members = config["ensemble_members"]

SAVE_DIR = f"/glade/work/awells/air_quality/{model}/pm25/file_paths/"

path_config = get_path_config(model, scenario)

for ens_num in ensemble_members:
    # Get file path
    dir_path = path_config["dir_template"].format(ens=f"{ens_num:02d}")
    search = os.path.join(dir_path, path_config["search_term"]).format(ens=f"{ens_num:02d}")

    # Find file list
    files = sorted(glob.glob(search))
    print(files[0])
    print(files[-1])

    # Save file list
    out_file = f"file_list_PM25_{scenario}_{ens_num:02d}.json"
    save_file_list_with_metadata(model, scenario, ens_num, files, SAVE_DIR, out_file)

print("All processing complete.")

/glade/campaign/collections/rda/data/d651059/ARISE-SAI-1.5/b.e21.BW.f09_g17.SSP245-G6-1p5K-SAI.001/atm/proc/tseries/month_1/b.e21.BW.f09_g17.SSP245-G6-1p5K-SAI.001.cam.h0.PM25.203501-208412.nc
/glade/campaign/collections/rda/data/d651059/ARISE-SAI-1.5/b.e21.BW.f09_g17.SSP245-G6-1p5K-SAI.001/atm/proc/tseries/month_1/b.e21.BW.f09_g17.SSP245-G6-1p5K-SAI.001.cam.h0.PM25.203501-208412.nc
/glade/campaign/collections/rda/data/d651059/ARISE-SAI-1.5/b.e21.BW.f09_g17.SSP245-G6-1p5K-SAI.002/atm/proc/tseries/month_1/b.e21.BW.f09_g17.SSP245-G6-1p5K-SAI.002.cam.h0.PM25.203501-208412.nc
/glade/campaign/collections/rda/data/d651059/ARISE-SAI-1.5/b.e21.BW.f09_g17.SSP245-G6-1p5K-SAI.002/atm/proc/tseries/month_1/b.e21.BW.f09_g17.SSP245-G6-1p5K-SAI.002.cam.h0.PM25.203501-208412.nc
/glade/campaign/collections/rda/data/d651059/ARISE-SAI-1.5/b.e21.BW.f09_g17.SSP245-G6-1p5K-SAI.003/atm/proc/tseries/month_1/b.e21.BW.f09_g17.SSP245-G6-1p5K-SAI.003.cam.h0.PM25.203501-208412.nc
/glade/campaign/collections/rda/dat